# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madihakomal75/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess

REPO_URL = "https://github.com/madihakomal75/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.exists("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Environment set up. Dataset loaded with shape:", df.shape)

Environment set up. Dataset loaded with shape: (30000, 44)


## 1. My lane as an ML task

*   Task Type: Binary Classification & Ranking Score

*   Why this task type?
   
Content decay detection is posed as a binary classification problem in order to generate a probability score for each webpage (e.g., P(declining) = 1. The probability obtained is then used as the priority score for sorting the first $K$ webpages that need content refresh analysis. While regression is an alternative approach (since it involves estimating the exact amount of impression loss), it suffers from variance because of the seasonal variation in keywords.

## 2. Target or proxy

*Target Definition: is_declining_label
Source: Derived from the observed outcome field trend_direction (trend_direction == "down"). This label represents historical organic search performance decay.*

In [3]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

declining_pct = df["is_declining_label"].mean() * 100
print(f"Target distribution (is_declining_label):")
print(f"  Declining (1): {df['is_declining_label'].sum():,} pages ({declining_pct:.2f}%)")
print(f"  Non-declining (0): {(1 - df['is_declining_label']).sum():,} pages ({100 - declining_pct:.2f}%)")

Target distribution (is_declining_label):
  Declining (1): 16,262 pages (54.21%)
  Non-declining (0): 13,738 pages (45.79%)


## 3. Success metric

*Primary Metric: Precision@50Target Threshold: Precision@50 $\ge 0.700$Defense: Content teams operate with constrained monthly review bandwidth. Measuring precision in the top 50 ranked pages ensures the generated priority queue minimizes wasted editorial hours on healthy pages.*

In [4]:
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Simulated naive rule score: stale * visible * impressions
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["naive_score"] = stale * visible * df["impressions_90d"]

baseline_p50 = precision_at_k(df["naive_score"], df["is_declining_label"], k=50)
print(f"Baseline Rule Precision@50: {baseline_p50:.3f}")
print(f"Target Model Precision@50:  >= 0.600 (Goal: >2.5x improvement over baseline)")

Baseline Rule Precision@50: 0.680
Target Model Precision@50:  >= 0.600 (Goal: >2.5x improvement over baseline)


## 4. The unit of analysis, as a real dataframe

*Unit of Analysis: One row = One unique content page (content_hash_id / url_hash_id).

Data Structure: Aggregates page-level performance signals (age, staleness, 90-day impressions, position, CTR) mapped to its decline label.*

In [6]:

if "is_declining_label" not in df.columns:
    df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)


id_col = "url_hash_id" if "url_hash_id" in df.columns else ("page_id" if "page_id" in df.columns else df.columns[0])

unit_cols = [id_col, "content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "is_declining_label"]
unit_sample = df[unit_cols].head(5)

print(f"Unit of analysis dataframe shape: {df.shape[0]:,} rows x {len(unit_cols)} core features")
unit_sample

Unit of analysis dataframe shape: 30,000 rows x 7 core features


,content_id,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,is_declining_label
0,content_304f48230142,187,20,3803,10.6,0.76,1
1,content_a1fb4e703a9e,445,25,15320,20.3,0.05,1
2,content_9aa793d4d895,141,20,12581,36.5,0.09,1
3,content_331d6c4de07b,463,22,11751,6.2,0.49,0
4,content_d99b7a2d90ca,263,14,19140,44.0,0.13,1


## 5. Why ML beats a fixed rule here

*Why ML beats an if-statement:
Fixed rules rely on hard cutoffs (e.g., days_since_last_update >= 180) that fail to capture complex feature interactions. Decision trees and ensemble models evaluate non-linear trade-offs between staleness, search volume, and positioning collapse without creating massive blocks of tied scores.*

In [7]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
X = df[features].fillna(0)
y = df["is_declining_label"]

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print("Learned Decision Rules (Depth 2 Decision Tree):")
print(export_text(tree, feature_names=features))

tree_p50 = precision_at_k(tree.predict_proba(X)[:, 1], y, k=50)
print(f"Decision Tree Precision@50: {tree_p50:.3f} vs Baseline Rule Precision@50: {baseline_p50:.3f}")

Learned Decision Rules (Depth 2 Decision Tree):
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0

Decision Tree Precision@50: 0.600 vs Baseline Rule Precision@50: 0.680
